# 🧠 Notebook 3 — Modèle U-Net & Entraînement
**ISIC Project | Segmentation de Lésions Cutanées**

Ce notebook couvre :
- ✅ Architecture U-Net from scratch
- ✅ Fonction de perte (Dice + BCE)
- ✅ **Reprise automatique depuis le dernier checkpoint**
- ✅ Sauvegarde du meilleur modèle
- ✅ Courbes d'apprentissage

## 🔗 1 — Setup & Connexion Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_PATH  = '/content/drive/MyDrive/ISIC_Project'
IMAGES_PATH   = os.path.join(PROJECT_PATH, 'data', 'Images')
MASQUES_PATH  = os.path.join(PROJECT_PATH, 'data', 'Masques')
SRC_PATH      = os.path.join(PROJECT_PATH, 'src')
OUTPUTS_PATH  = os.path.join(PROJECT_PATH, 'outputs')

# Chemin du checkpoint de reprise (sauvegardé à chaque epoch)
CHECKPOINT_PATH = os.path.join(OUTPUTS_PATH, 'checkpoint_last.pth')
# Chemin du meilleur modèle (sauvegardé uniquement si meilleur Dice)
BEST_MODEL_PATH = os.path.join(OUTPUTS_PATH, 'best_unet_isic.pth')

sys.path.append(SRC_PATH)
os.makedirs(OUTPUTS_PATH, exist_ok=True)

!pip install -q albumentations opencv-python-headless scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Setup OK | Device : {device}')

## 🔍 2 — Chargement des fichiers (filtre images uniquement)

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

all_images = sorted([p for p in Path(IMAGES_PATH).glob('*.*')
                     if p.suffix.lower() in IMAGE_EXTENSIONS])
all_masks  = sorted([p for p in Path(MASQUES_PATH).glob('*.*')
                     if p.suffix.lower() in IMAGE_EXTENSIONS])

assert len(all_images) == len(all_masks), \
    f'❌ Nombre images ({len(all_images)}) != masques ({len(all_masks)})'
print(f'✅ {len(all_images)} paires image/masque chargées')

## 📦 3 — Dataset & DataLoaders

In [ ]:
IMG_SIZE   = 256
BATCH_SIZE = 8

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ColorJitter(p=0.4),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class ISICDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']
            mask = out['mask'].unsqueeze(0)
        return img.float(), mask.float()

train_imgs, temp_imgs, train_msks, temp_msks = train_test_split(
    all_images, all_masks, test_size=0.3, random_state=42)
val_imgs, test_imgs, val_msks, test_msks = train_test_split(
    temp_imgs, temp_msks, test_size=0.5, random_state=42)

train_loader = DataLoader(ISICDataset(train_imgs, train_msks, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ISICDataset(val_imgs,   val_msks,   val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train : {len(train_loader)} batchs | Val : {len(val_loader)} batchs')

## 🧠 4 — Architecture U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2, 2)
        for f in features:
            self.downs.append(DoubleConv(in_channels, f))
            in_channels = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i//2]
            if x.shape != skip.shape:
                x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = self.ups[i+1](torch.cat([skip, x], dim=1))
        return self.final(x)

model = UNet().to(device)
print(f'✅ U-Net créé : {sum(p.numel() for p in model.parameters() if p.requires_grad):,} paramètres')

## 📉 5 — Fonction de perte (Dice + BCE)

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def dice_loss(self, pred, target, smooth=1e-6):
        pred  = torch.sigmoid(pred)
        inter = (pred * target).sum(dim=(2,3))
        union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
        return 1 - ((2*inter + smooth) / (union + smooth)).mean()
    def forward(self, pred, target):
        return 0.5 * self.bce(pred, target) + 0.5 * self.dice_loss(pred, target)

def dice_score(pred, target, threshold=0.5, smooth=1e-6):
    pred  = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return ((2*inter + smooth) / (union + smooth)).mean().item()

criterion = DiceBCELoss()
print('✅ DiceBCELoss définie !')

## 💾 6 — Fonctions Sauvegarde & Reprise du Checkpoint

> **Si la connexion coupe**, relance simplement ce notebook depuis le début.
> L'entraînement reprendra automatiquement depuis la dernière epoch sauvegardée.

In [ ]:
def save_checkpoint(epoch, model, optimizer, scheduler, history, best_dice, path):
    """Sauvegarde l'état complet de l'entraînement sur Drive."""
    torch.save({
        'epoch'        : epoch,
        'model_state'  : model.state_dict(),
        'optim_state'  : optimizer.state_dict(),
        'sched_state'  : scheduler.state_dict(),
        'history'      : history,
        'best_dice'    : best_dice,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler):
    """Charge le checkpoint et retourne l'epoch de reprise + l'historique."""
    ckpt = torch.load(path, map_location=device)
    model.state_dict()
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optim_state'])
    scheduler.load_state_dict(ckpt['sched_state'])
    return ckpt['epoch'], ckpt['history'], ckpt['best_dice']


print('✅ Fonctions save/load checkpoint prêtes !')

## 🚀 7 — Entraînement avec reprise automatique

In [ ]:
EPOCHS = 30
LR     = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5, verbose=True)

# ── Reprise depuis checkpoint si disponible ──────────────────────────────────
if os.path.exists(CHECKPOINT_PATH):
    start_epoch, history, best_dice = load_checkpoint(
        CHECKPOINT_PATH, model, optimizer, scheduler)
    start_epoch += 1   # on repart à l'epoch suivante
    print(f'🔄 Reprise depuis l\'epoch {start_epoch} '
          f'(meilleur Dice jusqu\'ici : {best_dice:.4f})')
else:
    start_epoch = 1
    best_dice   = 0.0
    history     = {'train_loss': [], 'val_loss': [],
                   'train_dice': [], 'val_dice': []}
    print('🆕 Démarrage d\'un nouvel entraînement')

# ── Boucle d'entraînement ────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):

    # ── Train ──
    model.train()
    train_loss, train_dice_val = 0.0, 0.0
    for imgs, masks in tqdm(train_loader,
                            desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        train_loss     += loss.item()
        train_dice_val += dice_score(preds, masks)
    train_loss     /= len(train_loader)
    train_dice_val /= len(train_loader)

    # ── Validation ──
    model.eval()
    val_loss, val_dice_val = 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            val_loss     += criterion(preds, masks).item()
            val_dice_val += dice_score(preds, masks)
    val_loss     /= len(val_loader)
    val_dice_val /= len(val_loader)

    scheduler.step(val_loss)

    # ── Mise à jour historique ──
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_dice_val)
    history['val_dice'].append(val_dice_val)

    # ── Sauvegarde checkpoint à CHAQUE epoch sur Drive ──
    save_checkpoint(epoch, model, optimizer, scheduler,
                    history, best_dice, CHECKPOINT_PATH)

    # ── Sauvegarde meilleur modèle ──
    if val_dice_val > best_dice:
        best_dice = val_dice_val
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        # Met aussi à jour le best_dice dans le checkpoint
        save_checkpoint(epoch, model, optimizer, scheduler,
                        history, best_dice, CHECKPOINT_PATH)
        print(f'💾 Epoch {epoch:02d} | Loss: {train_loss:.4f} '
              f'| Val Loss: {val_loss:.4f} '
              f'| Val Dice: {val_dice_val:.4f} ⬆️  BEST')
    else:
        print(f'   Epoch {epoch:02d} | Loss: {train_loss:.4f} '
              f'| Val Loss: {val_loss:.4f} '
              f'| Val Dice: {val_dice_val:.4f}')

print(f'\n✅ Entraînement terminé ! Meilleur Dice Val : {best_dice:.4f}')
print(f'💾 Checkpoint sauvegardé : {CHECKPOINT_PATH}')
print(f'💾 Meilleur modèle       : {BEST_MODEL_PATH}')

## ℹ️ Comment reprendre après une coupure ?

1. **Reconnecte-toi à Colab** et rouvre ce notebook
2. **Exécute toutes les cellules depuis le début** (Setup → Dataset → U-Net → Loss)
3. La cellule d'entraînement détecte automatiquement `checkpoint_last.pth` sur Drive
4. L'entraînement **reprend exactement à l'epoch suivante** sans rien perdre ✅

```
🔄 Reprise depuis l'epoch 12 (meilleur Dice jusqu'ici : 0.8234)
```

> **Fichiers sauvegardés sur Drive :**
> - `outputs/checkpoint_last.pth` → reprise complète (epoch, poids, optimizer, scheduler, historique)
> - `outputs/best_unet_isic.pth`  → meilleur modèle uniquement (pour l'évaluation finale)

## 📊 8 — Courbes d'apprentissage

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss', color='royalblue')
ax1.plot(history['val_loss'],   label='Val Loss',   color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Courbe de perte'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_dice'], label='Train Dice', color='royalblue')
ax2.plot(history['val_dice'],   label='Val Dice',   color='tomato')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice Score')
ax2.set_title('Courbe Dice Score'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'training_curves.png'), dpi=150)
plt.show()
print(f'📊 Epochs effectuées : {len(history["train_loss"])}/{EPOCHS}')
print(f'💾 Courbes sauvegardées : outputs/training_curves.png')

## ✅ Résumé
- U-Net entraîné avec Dice + BCE Loss
- **Checkpoint sauvegardé à chaque epoch** → reprise automatique après coupure
- Meilleur modèle : `outputs/best_unet_isic.pth`
- Courbes : `outputs/training_curves.png`

➡️ **Prochain notebook : `04_Evaluation_et_Predictions.ipynb`**